In [0]:
import importlib
import sys
from pathlib import Path
from pyspark.sql import functions as F

sys.dont_write_bytecode = True                       # keep __pycache__ out of the Git folder
REPO = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "configs" / "silver_config.json").exists())
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))
st = importlib.reload(importlib.import_module("src.silver.silver_transforms"))   # repo module, always latest code

CATALOG = "capstone9_smartmeter"
BRONZE, SILVER = f"{CATALOG}.bronze", f"{CATALOG}.silver"
cfg = st.load_config()
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {SILVER}")

def save(df, name):
    """Full deterministic rebuild of a Silver table from Bronze (idempotent)."""
    df.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{SILVER}.{name}")
    return spark.table(f"{SILVER}.{name}").count()

In [0]:
import pytest
for m in [m for m in sys.modules if m == "tests" or m.startswith("tests.")]:
    del sys.modules[m]                                  # always run the current test code
rc = pytest.main([str(REPO / "tests" / "silver"), "-v", "-p", "no:cacheprovider", "--rootdir", str(REPO)])
assert rc == 0, "Silver unit tests failed"

In [0]:
cols = ["rule_id", "entity", "column", "description", "check", "severity", "action"]
rules = spark.createDataFrame([tuple(r.get(c) for c in cols) for r in cfg["dq_rules"]],
                              "rule_id string, entity string, column_name string, description string, check_sql string, severity string, expected_action string")
print("dq_rule_catalogue:", save(rules, "dq_rule_catalogue"))
display(spark.table(f"{SILVER}.dq_rule_catalogue").orderBy("rule_id"))

In [0]:
meters, subs = spark.table(f"{BRONZE}.bronze_meter_id"), spark.table(f"{BRONZE}.bronze_substation_id")

rev = st.process_readings(spark.table(f"{BRONZE}.primary_events_raw"), meters, subs, cfg)
print("meter_readings:", save(st.silver_readings(rev), "meter_readings"))
readings = spark.table(f"{SILVER}.meter_readings")

sev = st.process_secondary(spark.table(f"{BRONZE}.secondary_events_raw"), readings, subs, cfg)
print("secondary_events:", save(st.silver_secondary(sev), "secondary_events"))

In [0]:
uev = st.process_status_updates(spark.table(f"{BRONZE}.status_updates_raw"), readings, cfg)
print("status_history:", save(st.build_status_history(readings, uev), "status_history"))
print("status_current:", save(st.status_current(spark.table(f"{SILVER}.status_history")), "status_current"))

In [0]:
quarantine = (st.quarantine_records(rev, "primary_events_raw", ["reading_id"], st.READING_COLS)
    .unionByName(st.quarantine_records(sev, "secondary_events_raw", ["event_id"], st.SECONDARY_COLS))
    .unionByName(st.quarantine_records(uev, "status_updates_raw", ["reading_id", "source_sequence"], st.STATUS_COLS)))
print("dq_quarantine:", save(quarantine.withColumn("quarantined_at", F.current_timestamp()), "dq_quarantine"))

dups = (st.duplicate_audit(rev, "primary_events_raw", ["reading_id"])
    .unionByName(st.duplicate_audit(sev, "secondary_events_raw", ["event_id"]))
    .unionByName(st.duplicate_audit(uev, "status_updates_raw", ["reading_id", "seq"])))
print("dq_duplicate_audit:", save(dups, "dq_duplicate_audit"))

In [0]:
iv = cfg["interval"]
print("meter_intervals:", save(st.build_expected_intervals(rev, readings, iv["interval_minutes"], iv["tolerance_minutes"]), "meter_intervals"))
print("meter_daily_energy:", save(st.meter_daily_energy(spark.table(f"{SILVER}.meter_intervals"), readings), "meter_daily_energy"))
print("daily_energy_reconciliation:", save(st.reconcile_daily(rev, spark.table(f"{SILVER}.meter_daily_energy")), "daily_energy_reconciliation"))

In [0]:
%sql
SELECT table_name, count(*) AS columns, concat_ws(', ', collect_list(concat(column_name, ' ', data_type))) AS schema
FROM capstone9_smartmeter.information_schema.columns
WHERE table_schema = 'silver'
GROUP BY table_name ORDER BY table_name

In [0]:
%sql
WITH s AS (
  SELECT 'primary_events_raw' AS source, (SELECT count(*) FROM capstone9_smartmeter.bronze.primary_events_raw) AS bronze_rows,
         (SELECT count(*) FROM capstone9_smartmeter.silver.meter_readings) AS silver_rows
  UNION ALL SELECT 'secondary_events_raw', (SELECT count(*) FROM capstone9_smartmeter.bronze.secondary_events_raw),
         (SELECT count(*) FROM capstone9_smartmeter.silver.secondary_events)
  UNION ALL SELECT 'status_updates_raw', (SELECT count(*) FROM capstone9_smartmeter.bronze.status_updates_raw),
         (SELECT count(*) - count_if(record_source = 'meter_readings') FROM capstone9_smartmeter.silver.status_history))
SELECT s.source, bronze_rows, silver_rows,
       coalesce(d.dropped, 0) AS duplicates_dropped, coalesce(q.rejected, 0) AS quarantined,
       bronze_rows - silver_rows - coalesce(d.dropped, 0) - coalesce(q.rejected, 0) AS unexplained
FROM s
LEFT JOIN (SELECT source_table, sum(copies_dropped) dropped FROM capstone9_smartmeter.silver.dq_duplicate_audit GROUP BY 1) d ON d.source_table = s.source
LEFT JOIN (SELECT source_table, count(*) rejected FROM capstone9_smartmeter.silver.dq_quarantine GROUP BY 1) q ON q.source_table = s.source

In [0]:
%sql
SELECT source_table, record_key, dq_failed_rules, dq_warnings, raw_record, _input_file_name
FROM capstone9_smartmeter.silver.dq_quarantine
ORDER BY source_table, record_key

In [0]:
%sql
SELECT reading_id, version, source_sequence, status, reported_previous_status, effective_from, effective_to,
       is_current, arrived_late, is_out_of_order, status_chain_ok, record_source
FROM capstone9_smartmeter.silver.status_history
WHERE reading_id IN (SELECT reading_id FROM capstone9_smartmeter.silver.status_history WHERE version > 1 ORDER BY reading_id LIMIT 5)
ORDER BY reading_id, version

In [0]:
%sql
SELECT count(*) AS days, count_if(status = 'PASS') AS days_pass, sum(bronze_rows) AS bronze_rows, sum(kept_rows) AS silver_rows,
       sum(duplicate_dropped_rows) AS duplicate_rows, sum(quarantined_rows) AS quarantined_rows,
       round(sum(bronze_kwh), 3) AS bronze_kwh, round(sum(kept_kwh), 3) AS silver_kwh, round(sum(daily_energy_kwh), 3) AS daily_total_kwh,
       round(sum(quarantined_kwh), 3) AS quarantined_kwh, sum(missing_intervals) AS missing_intervals,
       (SELECT count(*) FROM capstone9_smartmeter.silver.meter_intervals) AS expected_intervals,
       round(sum(missing_intervals) / (SELECT count(*) FROM capstone9_smartmeter.silver.meter_intervals), 5) AS missing_interval_rate
FROM capstone9_smartmeter.silver.daily_energy_reconciliation

In [0]:
def t(name):
    return spark.table(f"{SILVER}.{name}")


hist, cur = t("status_history"), t("status_current")
max_seq = hist.groupBy("reading_id").agg(F.max("source_sequence").alias("m"))
checks = {
    "Readings: 11,540 = 11,465 kept + 40 duplicates + 35 quarantined":
        (t("meter_readings").count(), rev.filter("outcome = 'DUPLICATE_DROPPED'").count(), rev.filter("outcome = 'QUARANTINED'").count()) == (11465, 40, 35),
    "Silver reading_id unique":                t("meter_readings").select("reading_id").distinct().count() == t("meter_readings").count(),
    "Raw values preserved (raw_consumption_kwh casts back)":
        t("meter_readings").filter("try_cast(raw_consumption_kwh AS DOUBLE) <> consumption_kwh").count() == 0,
    "Every quarantined row has a rule ID":      t("dq_quarantine").filter("size(dq_failed_rules) = 0").count() == 0,
    "One current status per reading":           cur.count() == cur.select("reading_id").distinct().count() == t("meter_readings").count(),
    "Current status = highest source_sequence": cur.join(max_seq, "reading_id").filter("source_sequence <> m").count() == 0,
    "Status rows: 520 = 518 applied + 2 quarantined (parent reading quarantined)":
        (hist.filter("record_source = 'status_updates'").count(), uev.filter("outcome = 'QUARANTINED'").count()) == (518, 2),
    "Missing intervals = 35 quarantined readings": t("meter_intervals").filter("is_missing").count() == 35,
    "Daily reconciliation PASS on every day":   t("daily_energy_reconciliation").filter("status <> 'PASS'").count() == 0,
}
for name, ok in checks.items():
    print("PASS" if ok else "FAIL", "|", name)
assert all(checks.values()), "Day 2 data checks failed"